In [3]:

%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [4]:
%pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="")

project = rf.workspace("").project("")

dataset = project.version().download("")


loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 63.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to My-First-Project-12 in yolov8:: 100%|██████████| 499/499 [00:00<00:00, 2703.04it/s]


In [ ]:
from pathlib import Path
from PIL import Image


def parse_yolo_pose_line(line):
    parts = line.strip().split()
    if len(parts) < 5:
        raise ValueError(f"Invalid label line: {line}")
    class_id = int(parts[0])
    x_center = float(parts[1])
    y_center = float(parts[2])
    width = float(parts[3])
    height = float(parts[4])
    keypoints = []
    for i in range(5, len(parts), 3):
        if i + 2 >= len(parts):
            break
        keypoints.append((float(parts[i]), float(parts[i + 1]), int(float(parts[i + 2]))))
    return {
        "class_id": class_id,
        "x_center": x_center,
        "y_center": y_center,
        "width": width,
        "height": height,
        "keypoints": keypoints,
    }


def yolo_to_absolute(label, img_w, img_h):
    x_center = label["x_center"] * img_w
    y_center = label["y_center"] * img_h
    width = label["width"] * img_w
    height = label["height"] * img_h
    x1 = x_center - width / 2
    y1 = y_center - height / 2
    x2 = x_center + width / 2
    y2 = y_center + height / 2
    return x1, y1, x2, y2


def absolute_to_yolo(x1, y1, x2, y2, tile_w, tile_h):
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    x_center = x1 + width / 2
    y_center = y1 + height / 2
    return x_center / tile_w if tile_w else 0.0, y_center / tile_h if tile_h else 0.0, width / tile_w if tile_w else 0.0, height / tile_h if tile_h else 0.0


def clip(value, low, high):
    return max(low, min(high, value))


def tile_label_keypoints(label, img_w, img_h, left, top, tile_w, tile_h):
    tiled_keypoints = []
    for x_rel, y_rel, v in label["keypoints"]:
        x_abs = x_rel * img_w
        y_abs = y_rel * img_h
        x_tile = x_abs - left
        y_tile = y_abs - top
        if x_tile < 0 or x_tile > tile_w or y_tile < 0 or y_tile > tile_h:
            tiled_keypoints.append((clip(x_tile, 0.0, tile_w), clip(y_tile, 0.0, tile_h), 0))
        else:
            tiled_keypoints.append((x_tile, y_tile, v))
    return [(x / tile_w if tile_w else 0.0, y / tile_h if tile_h else 0.0, v) for x, y, v in tiled_keypoints]


def tile_dataset(image_dir, label_dir, output_dir, slice_size=640, overlap_ratio=0.2):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)
    output_image_dir = Path(output_dir) / "images"
    output_label_dir = Path(output_dir) / "labels"
    output_image_dir.mkdir(parents=True, exist_ok=True)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(image_dir.glob("*.jpg")) + sorted(image_dir.glob("*.png"))
    tile_step = max(1, int(slice_size * (1.0 - overlap_ratio)))
    tile_count = 0

    for image_path in images:
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            print(f"Skipping {image_path.name}: missing label file")
            continue

        image = Image.open(image_path)
        img_w, img_h = image.size
        with open(label_path, "r", encoding="utf-8") as f:
            labels = [parse_yolo_pose_line(line) for line in f if line.strip()]

        for top in range(0, img_h, tile_step):
            for left in range(0, img_w, tile_step):
                right = min(left + slice_size, img_w)
                bottom = min(top + slice_size, img_h)
                tile_w = right - left
                tile_h = bottom - top
                if tile_w <= 0 or tile_h <= 0:
                    continue

                tile_labels = []
                for label in labels:
                    x1, y1, x2, y2 = yolo_to_absolute(label, img_w, img_h)
                    inter_x1 = max(x1, left)
                    inter_y1 = max(y1, top)
                    inter_x2 = min(x2, right)
                    inter_y2 = min(y2, bottom)
                    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
                        continue

                    bx1 = inter_x1 - left
                    by1 = inter_y1 - top
                    bx2 = inter_x2 - left
                    by2 = inter_y2 - top
                    x_center, y_center, width, height = absolute_to_yolo(bx1, by1, bx2, by2, tile_w, tile_h)
                    keypoints = tile_label_keypoints(label, img_w, img_h, left, top, tile_w, tile_h)

                    tile_labels.append({
                        "class_id": label["class_id"],
                        "x_center": x_center,
                        "y_center": y_center,
                        "width": width,
                        "height": height,
                        "keypoints": keypoints,
                    })

                if not tile_labels:
                    continue

                tile_name = f"{image_path.stem}_tile_{top}_{left}"
                tile_image = image.crop((left, top, right, bottom))
                tile_image.save(output_image_dir / f"{tile_name}.jpg")

                with open(output_label_dir / f"{tile_name}.txt", "w", encoding="utf-8") as out_f:
                    for label in tile_labels:
                        values = [
                            str(label["class_id"]),
                            f"{label['x_center']:.6f}",
                            f"{label['y_center']:.6f}",
                            f"{label['width']:.6f}",
                            f"{label['height']:.6f}",
                        ]
                        for kp_x, kp_y, kp_v in label["keypoints"]:
                            values.extend([f"{kp_x:.6f}", f"{kp_y:.6f}", str(kp_v)])
                        out_f.write(" ".join(values) + "\n")

                tile_count += 1

    print(f"Created {tile_count} tiled images in {output_image_dir}")
    return tile_count


def write_tiled_data_yaml(output_dir, class_names, nc=1, kpt_shape=[2, 3]):
    yaml_path = Path(output_dir) / "data_tiled.yaml"
    content = {
        "names": {i: name for i, name in enumerate(class_names)},
        "nc": nc,
        "kpt_shape": kpt_shape,
        "train": str((Path(output_dir) / "images").resolve()),
        "val": str((Path(output_dir) / "images").resolve()),
    }
    import yaml
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(content, f, sort_keys=False)
    print(f"Wrote tiled data config: {yaml_path}")
    return yaml_path


# Downloaded dataset location from Roboflow
local_dataset_dir = Path(getattr(dataset, "location", dataset))
print("Roboflow dataset downloaded to:", local_dataset_dir)
source_images = local_dataset_dir / "train" / "images"
source_labels = local_dataset_dir / "train" / "labels"
output_dir = Path("roboflow_yolov11_tiled") 
output_dir.mkdir(parents=True, exist_ok=True)

# Perform tiling
tiles = tile_dataset(
    image_dir=source_images,
    label_dir=source_labels,
    output_dir=output_dir,
    slice_size=640,
    overlap_ratio=0.2,
)

# Write a local data.yaml for the tiled dataset
try:
    import yaml
except ImportError:
    %pip install pyyaml
    import yaml

write_tiled_data_yaml(output_dir, class_names=["usaf_element"], nc=1, kpt_shape=[2, 3])
print("Tiling complete. Local tiled dataset stored in:", output_dir.resolve())


In [5]:
# 1. Mount Drive (do this first in a separate cell)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
from ultralytics import YOLO

RESUME_TRAINING = False

# Point directly to the file on your Drive
resume_path = ''

if RESUME_TRAINING:
    model = YOLO(resume_path)
    model.train(resume=True)
else:
    # Load the pretrained Pose model
    model = YOLO('yolo11m-pose-p2.yaml').load('yolo11m-pose.pt')
    # Train the model
    results = model.train(
        data='/content/your folder/data.yaml',
        epochs=100,
        imgsz=1920,
        project='your path',
        batch=9,
        device=0,
        task='pose',
        patience=30,
        save=True,
        degrees=0.0,
        kobj=2.0,
        mosaic=1.0,
        scale=0.5 
    )




Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=9, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/My-First-Project-12/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1920, int8=False, iou=0.7, keras=False, kobj=2.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,